# 9.1 章节介绍：性能优化高阶——负载均衡

[第7章](../07_performance_basics/README.md) 7.4 讲过均衡性的基本盘：**总耗时 = 最慢那个核算完的耗时**（木桶效应）。当时给的是理想化的贪心分核。本章进入真实推理场景——**带 section 的变长输入**，此时「每个核算多少」变得远比想象中复杂，负载均衡成为性能优化的最后一公里。

## 为什么负载均衡是「最后一公里」

优化到这个阶段，三引擎各自都快到位了：

- Cube 的 K 切分取了容量上限（[8.2/8.3](../08_performance_advanced/08.02_cube_k_split.ipynb)）；
- Vector 的 VF 指令精简、DN 选型都做完了（[8.4/8.5](../08_performance_advanced/08.04_vector_vf_opt.ipynb)）；
- Scalar 也上提增量化了（[8.6](../08_performance_advanced/08.06_scalar_opt.ipynb)）。

剩下的收益不在任何单个引擎里，而在**核间任务分配**：20 个核算 100 个任务，如果 19 个核算 1 个、剩下 1 个核算 81 个，那么无论单核多快，总耗时都由那 81 个任务的核决定。

## 本章的三个关键概念

| 概念 | 一句话解释 |
|--|--|
| **section** | KV 轴上的有效分段——推理是变长的：每条请求的有效 KV 长度不同（actualSeqLensKV），还可能有 shared prefix 共享前缀段 |
| **GS1** | 把 B×N×S1 展平后的全局 Q 行索引。「合轴/不合轴」指分核循环是把它当**一维轴**遍历，还是保持 **B、N 嵌套循环** |
| **切 S2** | 把 KV 方向也切到不同核（flash decoding）——Q 短 KV 长时唯一的并行度来源 |

## 两类分核策略（大纲主线）

| 策略 | 循环结构 | 特点 | 小节 |
|--|--|--|--|
| **GS1 不合轴 + 切 S2** | B、N 嵌套循环，S1 内层贪心累加 | 结构清晰、边界自然、跨 BN 组边界可能留缝 | [9.3](09.03_gs1_unmerged.ipynb) |
| **GS1 合轴 + 切 S2** | B×N×S1 展平成一维轴统一贪心 | 切分粒度更细、均衡上限更高、实现更重 | [9.4](09.04_gs1_merged.ipynb) |

两者都要解决同一个数学问题：**按有效基本块数（而不是名义 shape）把任务均分到核**。

## 本章学习路线

| 小节 | 主题 | 回答的问题 |
|--|--|--|
| [9.2 均衡问题的三层来源](09.02_imbalance_sources.ipynb) | shape 不齐、mask 无效块、section 变长 | 为什么名义 shape 均匀 ≠ 负载均匀？ |
| [9.3 GS1 不合轴 + 切 S2](09.03_gs1_unmerged.ipynb) | 贪心分核完整流程、六数组、flash decode 切 S2 | 嵌套循环下怎么分核？ |
| [9.4 GS1 合轴 + 切 S2](09.04_gs1_merged.ipynb) | 合轴的收益与代价、适用场景 | 什么时候值得展平成一维？ |
| [9.5 切 S2 的 softmax 归并](09.05_s2_merge.ipynb) | m/l 统计量、在线 softmax 归并、merge kernel | 切了 S2，softmax 怎么保证数值正确？ |
| [9.6 章节测验](09.06_chapter_test.ipynb) | 综合测验 | 检验本章掌握程度 |

## 前置知识

- [第7章 7.4 均衡性](../07_performance_basics/07.04_balance.ipynb)：CoreWeight、贪心分核、有效块统计——本章直接在其上展开
- [第5章 非量化 FA 开发](../05_flash_attn_non_quantized/README.md)：S2 外层循环与 softmax update——切 S2 是它的并行化推广
- [第2章 FlashAttention 原理](../02_fa_principle/README.md)：在线 softmax 的行级递推
